In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

print("GROQ_API_KEY configured:", bool(groq_api_key))

GROQ_API_KEY configured: True


In [3]:
from typing import List, TypedDict
import time

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langchain_huggingface import HuggingFaceEmbeddings

from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel

C:\Users\ce\AppData\Local\Temp\ipykernel_32020\2468276487.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\ce\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from langchain_groq import ChatGroq

# Replace this model if your Groq dashboard shows a different available model.
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
)

In [5]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = r"P:\ADVANCED_RAG\Corrective_RAG-CRAG-\documents\ML Book.pdf"  # local file path instead of a web URL

loader = PyPDFLoader(PDF_PATH)

raw_docs = loader.load()

print("Loaded documents:", len(raw_docs))
print("Source:", raw_docs[0].metadata.get("source"))

print("\nPreview:\n")
print(raw_docs[0].page_content[:1500])

Loaded documents: 564
Source: P:\ADVANCED_RAG\Corrective_RAG-CRAG-\documents\ML Book.pdf

Preview:

Aurélien Géron
Hands-On  
Machine Learning  
with Scikit-Learn  
& TensorFlow  
CONCEPTS, TOOLS, AND TECHNIQUES  
TO BUILD INTELLIGENT SYSTEMS
 D o w n l o a d   f r o m   f i n e l y b o o k   w w w . f i n e l y b o o k . c o m


In [6]:
len(raw_docs)

564

In [7]:
import re
import unicodedata


def clean_text(text: str) -> str:

    if not isinstance(text, str):
        text = str(text)

    # --------------------------------------------------
    # 1. Fix NULL-byte separated ASCII text
    # Example:
    # \x00D\x00o\x00w\x00n\x00l\x00o\x00a\x00d
    # becomes:
    # Download
    # --------------------------------------------------
    if "\x00" in text:

        # If NULL bytes are separating ASCII characters,
        # simply remove them.
        text = text.replace("\x00", "")

    # --------------------------------------------------
    # 2. Unicode normalization
    # --------------------------------------------------
    text = unicodedata.normalize("NFKC", text)

    # --------------------------------------------------
    # 3. Remove remaining control characters
    # --------------------------------------------------
    text = "".join(
        ch
        for ch in text
        if ch in ("\n", "\t")
        or not unicodedata.category(ch).startswith("C")
    )

    # --------------------------------------------------
    # 4. Normalize spaces
    # --------------------------------------------------
    text = re.sub(r"[ \t]+", " ", text)

    # --------------------------------------------------
    # 5. Normalize newlines
    # --------------------------------------------------
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [8]:
from langchain_core.documents import Document

clean_raw_docs = []

for doc in raw_docs:

    cleaned_text = clean_text(doc.page_content)

    clean_raw_docs.append(
        Document(
            page_content=cleaned_text,
            metadata=doc.metadata.copy()
        )
    )

print("Original documents:", len(raw_docs))
print("Clean documents:", len(clean_raw_docs))

Original documents: 564
Clean documents: 564


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = splitter.split_documents(clean_raw_docs)

print("Total chunks:", len(chunks))
print("\nFirst chunk preview:\n")
print(chunks[0].page_content[:900])

Total chunks: 1459

First chunk preview:

Aurélien Géron
Hands-On 
Machine Learning 
with Scikit-Learn 
& TensorFlow 
CONCEPTS, TOOLS, AND TECHNIQUES 
TO BUILD INTELLIGENT SYSTEMS
Download from finelybook www.finelybook.com


In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2410.53it/s]


In [ ]:
from langchain_chroma import Chroma

PERSIST_DIR = "./corrective_rag_db"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIR,
    collection_name="crag_demo",
)

print("Vectorstore created and persisted at:", PERSIST_DIR)

# Sanity check
query = "What is Agentic RAG?"
results = vectorstore.similarity_search(query, k=3)

print(f"\nTop {len(results)} results for query: '{query}'\n")
for i, doc in enumerate(results, 1):
    print(f"--- Result {i} ---")
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:300])
    print()

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [ ]:
class State(TypedDict):
    question: str
    docs: List[Document]

    strips: List[str]            # output of decomposition (sentence strips)
    kept_strips: List[str]       # after filtering (kept sentences)
    refined_context: str         # recomposed internal knowledge (joined kept_strips)

    answer: str

In [ ]:
# -----------------------------
# Sentence-level DECOMPOSER
# -----------------------------
def decompose_to_sentences(text: str) -> List[str]:
    text = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentences if len(s.strip()) > 20]




In [ ]:
decompose_to_sentences("""A transformer in deep learning is a 
type of model architecture that is particularly effective for 
processing sequential data, such as text. It utilizes mechanisms 
called self-attention and feedforward neural networks to weigh the 
importance of different parts of the input data, allowing it to capture 
long-range dependencies and relationships within the data. 
Unlike traditional recurrent neural networks (RNNs), 
transformers do not process data sequentially, 
which enables them to be more parallelizable and 
efficient in training. This architecture has become 
foundational in natural language processing tasks and 
has led to significant advancements in the field.""")

In [ ]:
# -----------------------------
# FILTER (LLM judge)
# -----------------------------
class KeepOrDrop(BaseModel):
    keep: bool

filter_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a strict relevance filter.\n"
            "Return keep=true only if the sentence directly helps answer the question.\n"
            "Use ONLY the sentence. Output JSON only.",
        ),
        ("human", "Question: {question}\n\nSentence:\n{sentence}"),
    ]
)

filter_chain = filter_prompt | llm.with_structured_output(KeepOrDrop)


In [ ]:
# -----------------------------
# REFINING (Decompose -> Filter -> Recompose)
# -----------------------------
def refine(state: State) -> State:

    q = state["question"]

    # Combine retrieved docs into one context string
    context = "\n\n".join(d.page_content for d in state["docs"]).strip()

    # 1) DECOMPOSITION: context -> sentence strips
    strips = decompose_to_sentences(context)

    # 2) FILTER: keep only relevant strips
    kept: List[str] = []
    
    for s in strips:
        if filter_chain.invoke({"question": q, "sentence": s}).keep:
            kept.append(s)

    # 3) RECOMPOSE: glue kept strips back together (internal knowledge)
    refined_context = "\n".join(kept).strip()

    return {
        "strips": strips,
        "kept_strips": kept,
        "refined_context": refined_context,
    }